<a href="https://colab.research.google.com/github/manasamorthad/DeepLearning/blob/main/dl_week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#week - 5

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/drive/MyDrive/DL_datasets/sign_mnist_train.csv")
test = pd.read_csv("/content/drive/MyDrive/DL_datasets/sign_mnist_test.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
X_train = train.drop("label",axis=1).values
y_train = train["label"].values

X_test = test.drop("label",axis=1).values
y_test = test["label"].values

X_train = X_train/255.0
X_test = X_test/255.0

X_train = X_train.reshape(-1,784)
X_test = X_test.reshape(-1,784)
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

def base_model():

    model = Sequential()
    model.add(Flatten(input_shape=(28,28,1)))
    model.add(Dense(128,activation='relu'))
    model.add(Dense(64,activation='relu'))
    model.add(Dense(25,activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

### Ensemble Modeling

Ensemble methods combine the predictions of several base estimators with a given learning algorithm to improve robustness over a single estimator and increase accuracy.

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score

models = []

# Ensure X_train and X_test are in the correct shape for base_model
# base_model expects (None, 28, 28, 1) as input due to the Flatten layer's input_shape.
if X_train.shape[1] == 784:
    X_train_for_base_model = X_train.reshape(-1, 28, 28, 1)
    X_test_for_base_model = X_test.reshape(-1, 28, 28, 1)
else:
    X_train_for_base_model = X_train
    X_test_for_base_model = X_test

print("Training ensemble models...")
for i in range(3):
    print(f"Training model {i+1}...")
    model = base_model()
    model.fit(X_train_for_base_model, y_train, epochs=5, batch_size=64, verbose=0)
    models.append(model)

print("Generating predictions...")
predictions = [m.predict(X_test_for_base_model) for m in models]

# Calculate the average of the predictions
avg_pred = np.mean(predictions, axis=0)

# Convert probabilities to class labels
final_pred = np.argmax(avg_pred, axis=1)

# Convert y_test back to class labels for accuracy calculation
y_test_labels = np.argmax(y_test, axis=1)

ensemble_accuracy = accuracy_score(y_test_labels, final_pred)
print(f"Ensemble Model Accuracy: {ensemble_accuracy:.4f}")

Training ensemble models...
Training model 1...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training model 2...
Training model 3...
Generating predictions...
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Ensemble Model Accuracy: 0.7075


### Dropout Regularization

Dropout is a regularization technique that randomly sets a fraction of input units to 0 at each update during training time, which helps prevent overfitting. Here, we'll build a new model with Dropout layers.

In [9]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Ensure the output layer has 25 units, as there are 25 classes (0-24)
model_dropout = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(25, activation='softmax') # Corrected to 25 units for 25 classes
])

model_dropout.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\nTraining model with Dropout...")
history_dropout = model_dropout.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))



Training model with Dropout...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.0528 - loss: 3.1852 - val_accuracy: 0.0604 - val_loss: 3.1028
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0699 - loss: 3.0766 - val_accuracy: 0.0519 - val_loss: 3.0937
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0720 - loss: 3.0364 - val_accuracy: 0.0818 - val_loss: 2.9300
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0760 - loss: 3.0178 - val_accuracy: 0.0711 - val_loss: 2.9281
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0759 - loss: 3.0085 - val_accuracy: 0.0718 - val_loss: 2.9064
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0805 - loss: 2.9946 - val_accuracy: 0.0781 - val_loss: 2.8964
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0791 - loss: 2.9939 - val_accuracy: 0.0634 - val_loss: 2.9125
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0806 - loss: 2.9862 - val_accuracy: 0.

1. L2

In [ ]:
from tensorflow.keras.regularizers import l2

model = Sequential([
Dense(128,activation='relu',input_shape=(784,),kernel_regularizer=l2(0.001)),
Dense(64,activation='relu',kernel_regularizer=l2(0.001)),
Dense(25,activation='softmax')
])

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

history = model.fit(X_train,y_train,epochs=10,batch_size=64,validation_data=(X_test,y_test))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3162 - loss: 2.4076 - val_accuracy: 0.4406 - val_loss: 1.9423
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5718 - loss: 1.4983 - val_accuracy: 0.5289 - val_loss: 1.5865
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6826 - loss: 1.1625 - val_accuracy: 0.5830 - val_loss: 1.4619
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7468 - loss: 0.9806 - val_accuracy: 0.6556 - val_loss: 1.3634
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8032 - loss: 0.8307 - val_accuracy: 0.6401 - val_loss: 1.3426
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8332 - loss: 0.7461 - val_accuracy: 0.6539 - val_loss: 1.3665
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8645 - loss: 0.6579 - val_accuracy: 0.6866 - val_loss: 1.2582
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8853 - loss: 0.5948 - val_accuracy: 0.


2. Dataset Augmentation


In [ ]:
X_train_img = X_train.reshape(-1,28,28,1)
X_test_img = X_test.reshape(-1,28,28,1)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
rotation_range=10,
width_shift_range=0.1,
height_shift_range=0.1,
zoom_range=0.1
)

datagen.fit(X_train_img)
model = base_model()

model.fit(
datagen.flow(X_train_img,y_train,batch_size=64),
epochs=10,
validation_data=(X_test_img,y_test)
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


429/429 ━━━━━━━━━━━━━━━━━━━━ 13s 26ms/step - accuracy: 0.1387 - loss: 2.8805 - val_accuracy: 0.2804 - val_loss: 2.2219
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.2887 - loss: 2.2605 - val_accuracy: 0.3854 - val_loss: 1.8906
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.3517 - loss: 2.0442 - val_accuracy: 0.4844 - val_loss: 1.6545
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.3883 - loss: 1.9110 - val_accuracy: 0.4962 - val_loss: 1.5128
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.4301 - loss: 1.7595 - val_accuracy: 0.5358 - val_loss: 1.3775
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.4687 - loss: 1.6398 - val_accuracy: 0.5629 - val_loss: 1.3012
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.4972 - loss: 1.5572 - val_accuracy: 0.5883 - val_loss: 1.2169
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.5238 - loss: 1.4730 - val_accuracy:

#parameter sharing

In [ ]:
from tensorflow.keras.layers import Conv2D,Flatten,MaxPooling2D

model = Sequential([
Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
MaxPooling2D(),
Flatten(),
Dense(64,activation='relu'),
Dense(25,activation='softmax')
])

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

model.fit(X_train_img,y_train,epochs=10,batch_size=64,validation_data=(X_test_img,y_test))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.4229 - loss: 2.0172 - val_accuracy: 0.5986 - val_loss: 1.3240
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7831 - loss: 0.7568 - val_accuracy: 0.7202 - val_loss: 0.9031
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8926 - loss: 0.3998 - val_accuracy: 0.7214 - val_loss: 0.8282
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9518 - loss: 0.2247 - val_accuracy: 0.7354 - val_loss: 0.7922
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9818 - loss: 0.1310 - val_accuracy: 0.7631 - val_loss: 0.7770
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9941 - loss: 0.0763 - val_accuracy: 0.7674 - val_loss: 0.8093
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9974 - loss: 0.0477 - val_accuracy: 0.7737 - val_loss: 0.8642
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9987 - loss: 0.0326 - val_accuracy: 0.

adding noise to input

In [ ]:
noise = np.random.normal(0,0.1,X_train.shape)

X_train_noisy = X_train + noise

model = base_model()

model.fit(X_train_noisy,y_train,epochs=10,batch_size=64,validation_data=(X_test,y_test))

early stopping

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Reshape X_train and X_test to match the input shape expected by base_model
X_train_reshaped = X_train.reshape(-1, 28, 28, 1)
X_test_reshaped = X_test.reshape(-1, 28, 28, 1)

model = base_model()

model.fit(
    X_train_reshaped, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_reshaped, y_test),
    callbacks=[early]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3596 - loss: 2.1194 - val_accuracy: 0.4527 - val_loss: 1.7076
Epoch 2/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6167 - loss: 1.2213 - val_accuracy: 0.6142 - val_loss: 1.2802
Epoch 3/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7264 - loss: 0.8641 - val_accuracy: 0.6272 - val_loss: 1.2262
Epoch 4/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8032 - loss: 0.6422 - val_accuracy: 0.6859 - val_loss: 1.0214
Epoch 5/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8504 - loss: 0.4988 - val_accuracy: 0.7075 - val_loss: 0.9806
Epoch 6/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8866 - loss: 0.3842 - val_accuracy: 0.7115 - val_loss: 0.9540
Epoch 7/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9136 - loss: 0.3008 - val_accuracy: 0.6977 - val_loss: 1.0445
Epoch 8/20
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9382 - loss: 0.2291 - val_accuracy: 0.

# week-6

In [10]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np

(X_train,y_train),(X_test,y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
X_train = X_train/255.0
X_test = X_test/255.0

X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense

model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(10,activation='softmax'))

model.compile(
optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
history = model.fit(
X_train,
y_train,
epochs=5,
batch_size=64,
validation_data=(X_test,y_test)
)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.9541 - loss: 0.1549 - val_accuracy: 0.9849 - val_loss: 0.0451
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9854 - loss: 0.0479 - val_accuracy: 0.9859 - val_loss: 0.0428
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9901 - loss: 0.0319 - val_accuracy: 0.9916 - val_loss: 0.0252
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9923 - loss: 0.0247 - val_accuracy: 0.9904 - val_loss: 0.0278
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9942 - loss: 0.0187 - val_accuracy: 0.9899 - val_loss: 0.0323


In [14]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/drive/MyDrive/DL_datasets/sign_mnist_train.csv")
test = pd.read_csv("/content/drive/MyDrive/DL_datasets/sign_mnist_test.csv")

In [15]:
X_train = train.drop("label",axis=1).values
y_train = train["label"].values

X_test = test.drop("label",axis=1).values
y_test = test["label"].values

X_train = X_train/255.0
X_test = X_test/255.0

X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

In [17]:
from tensorflow.keras.layers import Dropout

model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(25,activation='softmax'))
model.compile(
optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy']
)
history = model.fit(
X_train,
y_train,
epochs=10,
batch_size=64,
validation_data=(X_test,y_test)
)


Epoch 1/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.3575 - loss: 2.1292 - val_accuracy: 0.7296 - val_loss: 1.0450
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6813 - loss: 0.9815 - val_accuracy: 0.7759 - val_loss: 0.6777
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7975 - loss: 0.6131 - val_accuracy: 0.8238 - val_loss: 0.5469
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8501 - loss: 0.4477 - val_accuracy: 0.8383 - val_loss: 0.5230
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8891 - loss: 0.3314 - val_accuracy: 0.8466 - val_loss: 0.5300
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9072 - loss: 0.2751 - val_accuracy: 0.8451 - val_loss: 0.5477
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9254 - loss: 0.2217 - val_accuracy: 0.8525 - val_loss: 0.5558
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9361 - loss: 0.1881 - val_accuracy: 0.

In [18]:
from tensorflow.keras.layers import Dropout

model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation='relu'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(25,activation='softmax'))

In [19]:
model.compile(
optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy']
)
history = model.fit(
X_train,
y_train,
epochs=10,
batch_size=64,
validation_data=(X_test,y_test)
)

Epoch 1/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.4664 - loss: 1.6968 - val_accuracy: 0.8504 - val_loss: 0.5459
Epoch 2/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8504 - loss: 0.4382 - val_accuracy: 0.9129 - val_loss: 0.3121
Epoch 3/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9318 - loss: 0.1966 - val_accuracy: 0.9159 - val_loss: 0.2584
Epoch 4/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9595 - loss: 0.1169 - val_accuracy: 0.9395 - val_loss: 0.2241
Epoch 5/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9744 - loss: 0.0805 - val_accuracy: 0.9334 - val_loss: 0.2640
Epoch 6/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9781 - loss: 0.0636 - val_accuracy: 0.9505 - val_loss: 0.2215
Epoch 7/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9822 - loss: 0.0525 - val_accuracy: 0.9619 - val_loss: 0.2183
Epoch 8/10
429/429 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9856 - loss: 0.0430 - val_accuracy: 